# Deploy the Brand-Guidelines Checker Agent

![Agent deploy](imgs/agent_deploy.png)

## Objective
Deploy the single-turn retail brand-guidelines checker onto the managed Agent Runtime on the Gemini Enterprise Agent Platform (GEAP). The notebook is self-contained: it provisions prerequisites, stages the agent as a source package, and deploys.

The agent is packaged as a **container built from source** (not a pickled object): you ship the source files and the platform builds a container image with dependencies baked in, then runs it. Runtime revisions populate — which enables canary / traffic-split rollback later — and you ship the image you tested. This is the Level 2 / CI-CD-aligned packaging.

Documentation: [Deploy an agent](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/deploy-an-agent) · [Agent Registry](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/agent-registry)

## Set Variables

- `PROJECT`: Google Cloud project ID where the agent and ops resources are deployed.
- `REGION`: Google Cloud region for deployment (Agent Runtime, GEAP).
- `AGENT`: which agent this run targets; every agent lives in its own folder under `agents/`. Change this to demo another agent.
- `AGENT_DIR`: path to that agent's folder, passed to the shared scripts with `--agent-dir`.

In [1]:
import os

PROJECT = 'sandbox-401718'  # @param {type:"string"}
REGION = 'us-central1'      # @param {type:"string"}
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT
os.environ['GOOGLE_CLOUD_LOCATION'] = REGION

# Every agent lives under agents/; point the shared scripts at it with --agent-dir.
AGENT = 'brand-guidelines-checker'  # @param {type:"string"}
AGENT_DIR = f'../agents/{AGENT}'

# Provisioning is optional — the deploy below auto-provisions. To do it explicitly:
#   !agents-cli infra single-project --project $PROJECT --apply

In [2]:
# Provision prerequisites (OPTIONAL) — the deploy auto-provisions, so skip unless you want
# service accounts / IAM / APIs / telemetry set up imperatively (same set terraform/ declares).

# !agents-cli infra single-project --project $PROJECT --apply

## Create the Staging Bucket

The SDK stages the packaged agent to GCS before building the Agent Engine, so it needs a bucket. Staging is ephemeral build scratch (not agent-specific), so it's a single project-level bucket (`{project}-agent-staging`) shared by every agent — separate from the per-agent `{project}-{agent}-artifacts` buckets that hold eval outputs. The cell is idempotent (a no-op on re-runs).

In [3]:
# Staging bucket — shared, project-level build scratch for the SDK deploy. Idempotent.
import subprocess

STAGING_BUCKET = f'{PROJECT}-agent-staging'
BUCKET_URI = f'gs://{STAGING_BUCKET}'

_exists = (
    subprocess.run(
        ['gcloud', 'storage', 'buckets', 'describe', BUCKET_URI, '--project', PROJECT],
        capture_output=True,
        text=True,
    ).returncode
    == 0
)

if _exists:
    print(f'Bucket {BUCKET_URI} already exists — reusing.')
else:
    subprocess.run(
        [
            'gcloud',
            'storage',
            'buckets',
            'create',
            BUCKET_URI,
            '--project',
            PROJECT,
            '--location',
            REGION,
            '--uniform-bucket-level-access',
        ],
        check=True,
    )
    subprocess.run(
        ['gcloud', 'storage', 'buckets', 'update', BUCKET_URI, '--versioning'],
        check=True,
    )
    print(f'Created {BUCKET_URI}')

Bucket gs://sandbox-401718-agent-staging already exists — reusing.


## Deploy the Agent (source → container build)

Deploy the brand checker onto Agent Runtime by staging it as a source package and letting the platform build a container. The build bakes in telemetry and prompt/response capture so traces carry message content — required for GEAP Agent Evaluation (NB2).

Environment variables set on the build:
- `GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY=true`: telemetry + observability dashboard.
- `OTEL_SEMCONV_STABILITY_OPT_IN=gen_ai_latest_experimental` + `OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT=EVENT_ONLY`: write prompt/response into the `gen_ai` inference event the rater reads.
- `GOOGLE_CLOUD_LOCATION=global`: route model calls to the global endpoint, so global-only models like `gemini-3.5-flash` serve from this regional engine; the engine's own region stays via the runtime-set `GOOGLE_CLOUD_AGENT_ENGINE_LOCATION`.
- `AdkApp(..., enable_tracing=True)` + `OTEL_INSTRUMENTATION_GENAI_UPLOAD_*`: Cloud Trace spans + GCS payload storage.

Source-packaging notes:
- Source files must live under the notebook's working dir, and the package dir must be a valid Python module name. The agent folder (`brand-guidelines-checker`) has dashes, so we stage a dashless `_serving_build/` package.
- The container entrypoint must expose a wrapped `AdkApp` (a raw `LlmAgent` fails startup), so `app.py` wraps the agent; the instruction is baked into the image (`instruction.txt`).

Unlike pickle, this populates runtime revisions (each redeploy of versioned fields = a new revision) — what later enables canary / traffic-split rollback.

Note: `DEPLOY_WEAK_BASELINE = True` bakes a deliberately weak instruction on the incumbent model (`gemini-2.5-flash`) so the live agent is the drifted one NB2/NB3 react to. Set `False` for the healthy agent.

Note: on the first container deploy, if a prior pickle engine of the same name exists, delete it (teardown) first so this creates a clean source engine rather than converting deploy modes in place. Re-running afterward updates in place (new revision, same engine = lineage).

Documentation: [Deploy an agent](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/deploy-an-agent) · [Manage revisions and traffic](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/manage-revisions-and-traffic)

Note: the agent now uses the agents-cli **`app/` structure** (`app/agent.py` + `agents-cli-manifest.yaml`); the canonical deploy is **`agents-cli deploy`** — the default of `deployment/deploy_agent.py` (source/container + Agent Identity, revisions populate). This notebook keeps the deploy inline below to show the SDK `source_packages` mechanics and bake the demo's weak baseline — the *same* mechanism agents-cli uses.


In [4]:
# Stage the agent as a SOURCE package for a container build (the L2-style deploy).
# Pickle ships a frozen object; here we ship source and the platform builds an image
# (deps baked in) -> runtime revisions populate, and there's no unpickle version-skew.
import os, shutil, importlib.util, yaml

with open(f'{AGENT_DIR}/agent.yaml') as f:
    cfg = yaml.safe_load(f)
display_name = cfg.get('display_name') or cfg.get('name', AGENT)
description = cfg.get('description')
MODEL = cfg.get('model', 'gemini-2.5-flash')

# DEMO: bake a deliberately-weak instruction so the live (incumbent) agent is the drifted
# one NB2/NB3 react to. Set False for the healthy agent. (Same weak file NB3 uses.)
DEPLOY_WEAK_BASELINE = True  # @param {type:"boolean"}
os.environ['MODEL'] = MODEL
if DEPLOY_WEAK_BASELINE:
    os.environ['AGENT_INSTRUCTION_FILE'] = os.path.abspath(f'{AGENT_DIR}/eval/weak_instruction.txt')
    print('baking WEAK baseline (simulated drift) on', MODEL)
else:
    os.environ.pop('AGENT_INSTRUCTION_FILE', None)

# Single source of truth: import agent.py to resolve the exact name + model + instruction,
# then bake them into a dashless source package (the agent dir has dashes -> not a module).
spec = importlib.util.spec_from_file_location('agent_src', f'{AGENT_DIR}/app/agent.py')
_m = importlib.util.module_from_spec(spec); spec.loader.exec_module(_m)
BAKED_NAME = _m.root_agent.name          # valid identifier (e.g. brand_guidelines_checker)
BAKED_MODEL = _m.root_agent.model
BAKED_INSTRUCTION = _m.root_agent.instruction

BUILD_PKG = '_serving_build'   # valid module name, created under this notebook's cwd
shutil.rmtree(BUILD_PKG, ignore_errors=True); os.makedirs(BUILD_PKG)

# app.py = the container entrypoint. MUST expose a wrapped AdkApp ('app'); a raw LlmAgent
# fails startup ("missing query/stream_query"). MODEL overridable via env (NB3 candidate).
open(f'{BUILD_PKG}/app.py', 'w').write(
    'import os\n'
    'from google.adk.agents import Agent\n'
    'from vertexai import agent_engines\n'
    '_here = os.path.dirname(os.path.abspath(__file__))\n'
    f'MODEL = os.environ.get("MODEL", {BAKED_MODEL!r})\n'
    'with open(os.path.join(_here, "instruction.txt")) as f:\n'
    '    INSTRUCTION = f.read().strip()\n'
    f'root_agent = Agent(name={BAKED_NAME!r}, model=MODEL, instruction=INSTRUCTION)\n'
    f'app = agent_engines.AdkApp(agent=root_agent, app_name={BAKED_NAME!r}, enable_tracing=True)\n'
)
open(f'{BUILD_PKG}/instruction.txt', 'w').write(BAKED_INSTRUCTION)
open(f'{BUILD_PKG}/requirements.txt', 'w').write(open(f'{AGENT_DIR}/requirements.txt').read())
print('staged', BUILD_PKG, '->', os.listdir(BUILD_PKG))


baking WEAK baseline (simulated drift) on gemini-2.5-flash


/opt/conda/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_version" in LlmResponse has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_version" in Event has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


staged _serving_build -> ['requirements.txt', 'instruction.txt', 'app.py']


/opt/conda/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_code" in LlmAgentConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [5]:
# Deploy the source package -> Agent Runtime builds a container and registers a revision.
import vertexai
from vertexai import agent_engines
from google.genai import types as genai_types
from vertexai._genai.types.common import AgentEngineConfig
import vertexai._genai._agent_engines_utils as _aeu
from google.adk.agents import Agent as _Agent

vertexai.init(project=PROJECT, location=REGION, staging_bucket=BUCKET_URI)

# class_methods = the operation schema the source server registers. Source deploys require
# it explicitly; we generate the same set the SDK auto-creates, from a local AdkApp.
_local_app = agent_engines.AdkApp(
    agent=_Agent(name=BAKED_NAME, model=BAKED_MODEL, instruction=BAKED_INSTRUCTION),
    app_name=BAKED_NAME)
_spec = _aeu._generate_class_methods_spec_or_raise(
    agent=_local_app, operations=_local_app.register_operations())
class_methods = [_aeu._to_dict(s) for s in _spec]

_env = {
    'GOOGLE_CLOUD_LOCATION': 'global',  # model calls -> global endpoint (serves global-only models, e.g. 3.5)
    'GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY': 'true',
    'OTEL_SEMCONV_STABILITY_OPT_IN': 'gen_ai_latest_experimental',
    'OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT': 'EVENT_ONLY',
    'OTEL_INSTRUMENTATION_GENAI_UPLOAD_FORMAT': 'jsonl',
    'OTEL_INSTRUMENTATION_GENAI_COMPLETION_HOOK': 'upload',
    'OTEL_INSTRUMENTATION_GENAI_UPLOAD_BASE_PATH': f'{BUCKET_URI}/agent-payloads',
}

deploy_cfg = AgentEngineConfig(
    staging_bucket=BUCKET_URI,
    source_packages=[BUILD_PKG],
    entrypoint_module=f'{BUILD_PKG}.app',
    entrypoint_object='app',
    class_methods=class_methods,
    requirements_file=f'{BUILD_PKG}/requirements.txt',
    agent_framework='google-adk',
    env_vars=_env,
    display_name=display_name,
    description=description,
)

client = vertexai.Client(project=PROJECT, location=REGION,
                         http_options=genai_types.HttpOptions(api_version='v1beta1'))

# Upsert by display_name (preserve lineage = one engine, a new revision per deploy).
matches = [e for e in agent_engines.list()
           if (getattr(e, 'display_name', '') or '') == display_name]
if matches:
    client.agent_engines.update(name=matches[0].resource_name, config=deploy_cfg)
    print('updated in place (new revision):', matches[0].resource_name)
else:
    remote = client.agent_engines.create(config=deploy_cfg)
    name = (getattr(remote, 'api_resource', None) and remote.api_resource.name) or getattr(remote, 'name', None)
    print('deployed (new source engine):', name)


/opt/conda/lib/python3.10/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


updated in place (new revision): projects/757654702990/locations/us-central1/reasoningEngines/3538275147427348480


## Verify Components (deploy-time)

Confirm what exists the moment the deploy finishes — before the agent is ever called:
- **Agent Identity (SPIFFE)** is issued — Agent Runtime auto-assigns each agent a SPIFFE identity (X.509 cert, 24h, auto-renewed), mapped to the engine resource URI. You grant IAM to its `principal://…`; it can't be impersonated and has no long-lived keys.
- The revision is auto-registered in Agent Registry.

**Agent Identity & least-privilege.** The SPIFFE identity is auto-issued (nothing to configure), so the governance you own is least-privilege: the loop's automated eval/deploy calls run under a dedicated per-agent SA (`{agent}-sa`) with a scoped role allowlist — enforced statically in CI (`tests/test_identity.py`, NB5) and re-checked post-deploy in CD (NB6).

Telemetry (Cloud Trace spans, Cloud Logging entries) is verified later in *Verify Telemetry* — a trace span isn't written until the agent is invoked, so there's nothing to check here yet.

Documentation: [Agent Identity](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/agent-identity-overview) · [Agent Registry](https://docs.cloud.google.com/agent-registry/automatic-registration)


In [6]:
# Confirm auto-registration: deploying to Agent Runtime auto-registers the engine in
# Agent Registry (Preview/console-driven), so the practical check is that the engine is
# live in Agent Runtime. (Inline; mirrors deployment/register_agent.py.)
import vertexai
from vertexai import agent_engines


def _norm(s):
    return ''.join(c for c in s.lower() if c.isalnum())


vertexai.init(project=PROJECT, location=REGION)
matches = [
    (getattr(e, 'display_name', '') or '', e.resource_name)
    for e in agent_engines.list()
    if _norm(AGENT) in _norm(getattr(e, 'display_name', '') or '')
]
for d, r in matches:
    print('found:', d, '|', r)
print(
    'auto-registration OK — engine is live (and synced to Agent Registry).'
    if matches
    else 'no match yet — give it a moment and re-run.'
)
if len(matches) > 1:
    print(
        'WARNING: multiple matching engines — delete stale ones so NB2 targets the right one.'
    )

# Agent Identity & least-privilege — surface the identity anchors:
#  - SPIFFE is auto-issued, mapped to the engine resource URI below (grant IAM to its principal);
#  - the loop's automated calls use the per-agent least-privilege SA (verified by tests/test_identity.py).
for _d, _r in matches:
    print('identity anchor (SPIFFE maps to this resource):', _r)
print('least-privilege ops SA (per agent):', f'{AGENT}-sa@{PROJECT}.iam.gserviceaccount.com')


found: Brand Guidelines Checker | projects/757654702990/locations/us-central1/reasoningEngines/3538275147427348480
auto-registration OK — engine is live (and synced to Agent Registry).
identity anchor (SPIFFE maps to this resource): projects/757654702990/locations/us-central1/reasoningEngines/3538275147427348480
least-privilege ops SA (per agent): brand-guidelines-checker-sa@sandbox-401718.iam.gserviceaccount.com


## Call the Agent (sample prompt)

Send a piece of retail marketing copy to the deployed agent and print its structured JSON verdict. This also writes the first Cloud Trace span — traces are emitted on invocation, not at deploy, so the trace view stays empty until you run this.

The agent checks copy against the brand guidelines baked into its instruction (`agents/brand-guidelines-checker/agent.py`):
1. Tone is warm and plain-spoken. No hype words: *revolutionary*, *game-changing*, *best-in-class*, *unbeatable*.
2. Never promise specific savings amounts or use *guaranteed*.
3. Price claims must include "while supplies last" if a discount is named.
4. Use sentence case for headlines, never ALL CAPS.
5. Refer to customers as "you", never *consumers* or *users*.

It returns JSON only: `{ "verdict": "PASS"|"FAIL", "violations": [{"rule", "quote"}], "suggested_fix" }`.

The sample below is built to break every rule at once — an ALL-CAPS headline (rule 4), stacked hype words (rule 1), a `guaranteed` specific-dollar saving (rule 2), a named discount with no "while supplies last" (rule 3), and *consumers*/*users* instead of *you* (rule 5) — so the verdict should come back `FAIL` with each offending quote and a compliant rewrite.

In [7]:
# Sample call — resolve the deployed engine by name, send marketing copy, print the verdict.
import vertexai
from vertexai import agent_engines

vertexai.init(project=PROJECT, location=REGION)


def _norm(s):
    return ''.join(c for c in s.lower() if c.isalnum())


engine = next(
    e
    for e in agent_engines.list()
    if _norm(AGENT) in _norm(getattr(e, 'display_name', '') or '')
)

# Deliberately violates all five brand rules so every check lights up:
#   rule 4 ALL-CAPS headline | rule 1 hype words | rule 2 "guaranteed" + $ amount
#   rule 3 named discount, no "while supplies last" | rule 5 "consumers"/"users"
SAMPLE_PROMPT = (
    'MEGA SUMMER BLOWOUT SALE!!!\n\n'
    'Our revolutionary, game-changing blender is the best-in-class pick for '
    'smart consumers. Take 40% off today and save a guaranteed $50 — an '
    "unbeatable deal that savvy users won't find anywhere else."
)

session = engine.create_session(user_id='demo-user')
chunks = []
for event in engine.stream_query(
    user_id='demo-user', session_id=session['id'], message=SAMPLE_PROMPT
):
    for part in event.get('content', {}).get('parts', []):
        if part.get('text'):
            chunks.append(part['text'])

print('PROMPT:\n', SAMPLE_PROMPT, '\n')
print('VERDICT:\n', ''.join(chunks))

PROMPT:
 MEGA SUMMER BLOWOUT SALE!!!

Our revolutionary, game-changing blender is the best-in-class pick for smart consumers. Take 40% off today and save a guaranteed $50 — an unbeatable deal that savvy users won't find anywhere else. 

VERDICT:
 ```json
{
  "verdict": "FAIL",
  "violations": [
    {
      "rule": 1
    },
    {
      "rule": 2
    },
    {
      "rule": 3
    },
    {
      "rule": 4
    }
  ]
}
```


### Verify Telemetry (post-call)

Now that the sample call has invoked the agent, the telemetry paths can be confirmed — the pieces that don't exist until there's traffic:
- Logs route to Cloud Logging.
- A trace appears in Cloud Trace (written on the first invocation, which the sample call just triggered).

Easiest to confirm in the console, but you can also check from code. The cell below reads the most recent Cloud Logging entries for the deployed Agent Runtime — live proof the logging path is wired.

Note: reading logs requires `roles/logging.viewer` on the project; if you get `PERMISSION_DENIED`, grant that role to the identity running this notebook.

Documentation: [Tracing](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/tracing) · [Logging](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/logging)

In [8]:
# Recent Cloud Logging entries for the deployed Agent Runtime (GA, no extra deps).
# Empty in a brand-new project until the deploy above has produced log entries.
!gcloud logging read 'resource.type="aiplatform.googleapis.com/ReasoningEngine"' \
    --project $PROJECT --limit 5 --freshness 1d \
    --format='table(timestamp, severity, resource.labels.reasoning_engine_id)'

TIMESTAMP                    SEVERITY  REASONING_ENGINE_ID
2026-06-05T04:31:28.977861Z            3538275147427348480
2026-06-05T04:31:20.596803Z            3538275147427348480
2026-06-05T04:31:20.595674Z            3538275147427348480
2026-06-05T04:31:20.595567Z            3538275147427348480
2026-06-05T04:31:20.595545Z            3538275147427348480
